# 10 - Customer Lifetime Value Analysis

## Customer Intelligence Platform

IBM's dataset includes a CLTV column, but we correctly identified it as leakage-prone.
This notebook builds our own CLV estimates from first principles.

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from src.load_data import load_features
from src.config import TARGET, MODELS_DIR, BEST_MODEL_FILE

%matplotlib inline


In [ ]:
df = load_features()

# Load model for churn probabilities
model = joblib.load(BEST_MODEL_FILE)


## 1. Basic CLV

The simplest CLV estimate: total revenue generated to date.

```
Basic CLV = Monthly Charges x Tenure Months
```


In [ ]:
df["clv_basic"] = df["Monthly Charges"] * df["Tenure Months"]
print(f"Basic CLV Statistics:")
print(df["clv_basic"].describe())


## 2. Enhanced CLV

Adjusting for expected future revenue using churn probability:

```
Enhanced CLV = Monthly Charges x Expected Remaining Tenure x (1 - Churn Probability)
```

Where Expected Remaining Tenure is estimated from the overall retention rate.


In [ ]:
# Get churn probabilities from the model
from src.preprocessing import encode_features, split_data, scale_features

df_encoded = encode_features(df.copy())
feature_names_path = MODELS_DIR / "feature_names.json"
import json
with open(feature_names_path) as f:
    feature_names = json.load(f)

# Align features
X = df_encoded.drop(columns=[TARGET], errors="ignore")
for feat in feature_names:
    if feat not in X.columns:
        X[feat] = 0
X = X[feature_names]

# Scale
scaler = joblib.load(MODELS_DIR / "scaler.joblib")
num_cols = ["Tenure Months", "Monthly Charges", "Total Charges",
            "service_count", "revenue_per_month", "revenue_intensity",
            "tech_adoption_score", "risk_score", "contract_tenure"]
scale_cols = [c for c in num_cols if c in X.columns]
X[scale_cols] = scaler.transform(X[scale_cols])

df["churn_probability"] = model.predict_proba(X)[:, 1]
print(f"Churn probability computed for {len(df):,} customers")
print(f"Mean churn probability: {df['churn_probability'].mean():.2%}")


In [ ]:
# Expected remaining tenure (in months)
avg_monthly_retention_rate = 1 - df[TARGET].mean()  # ~73.5%
max_expected_months = 60  # 5 years

# Enhanced CLV
df["expected_remaining_tenure"] = np.clip(
    (1 / (1 - avg_monthly_retention_rate + 0.001)) * (1 - df["churn_probability"]),
    0, max_expected_months
)

df["clv"] = df["Monthly Charges"] * df["expected_remaining_tenure"]
print(f"\nEnhanced CLV Statistics:")
print(df["clv"].describe())


## 3. CLV Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["clv_basic"], bins=50, color="#3498db", alpha=0.7, edgecolor="white")
axes[0].set_title("Basic CLV Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("CLV ($)")
axes[0].set_ylabel("Count")

axes[1].hist(df["clv"], bins=50, color="#9b59b6", alpha=0.7, edgecolor="white")
axes[1].set_title("Enhanced CLV Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("CLV ($)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()


## 4. High Value + High Risk Identification

The most critical business output: customers who are both valuable AND at risk.


In [ ]:
df["clv_tier"] = pd.qcut(df["clv"], q=3, labels=["Low", "Medium", "High"])
df["risk_level"] = pd.cut(
    df["churn_probability"],
    bins=[0, 0.4, 0.7, 1.0],
    labels=["Low", "Medium", "High"]
)

# Cross-tabulation
priority_matrix = pd.crosstab(df["clv_tier"], df["risk_level"], margins=True)
priority_matrix


In [ ]:
# Priority list: High CLV + High Risk
priority_customers = df[
    (df["clv_tier"] == "High") & (df["risk_level"] == "High")
]
print(f"HIGH VALUE + HIGH RISK Customers: {len(priority_customers):,}")
print(f"Monthly Revenue at Risk: ${priority_customers['Monthly Charges'].sum():,.0f}")
print(f"Average CLV: ${priority_customers['clv'].mean():,.0f}")


## Summary

| CLV Type | Method | Use Case |
|----------|--------|----------|
| Basic CLV | Monthly x Tenure | Historical value |
| Enhanced CLV | Adjusted by churn probability | Forward-looking value |

### Priority Actions
1. **High CLV + High Risk**: Immediate intervention (premium retention package)
2. **High CLV + Medium Risk**: Proactive monitoring
3. **Medium CLV + High Risk**: Cost-effective retention offers
